<center><h1>Wang_Guanqi_HW3</h1></center>
<br>
<br>

Name: Guanqi Wang
<br>
Github Username: 
<br>
USC ID: 8047294413

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.neighbors import KNeighborsRegressor
from tornado.gen import multi
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
import os

Get the AReM Data Set

In [3]:
data_path = r'./AReM'

### (b) Test and Train Data

### (c) Feature Extraction

#### i. Research

Commonly used are Minimum, Maximum, Mean, Median, Standard Deviation, Variance, Range, IOR, Skewness, etc.

#### ii. Extraction

In [22]:
activities = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
time_series = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
features = ["min", "max", "mean",  "median", "std", "first_quartile", "third_quartile"]

all = []

for activity in activities:
    folder = os.path.join(data_path, activity)
    for file in os.listdir(folder):
        if not file.endswith('.csv'):
            continue
        file_path = os.path.join(folder, file)
        df = pd.read_csv(file_path, header=None,names=["time", 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23'], comment='#', sep=r'[,\s]+', engine='python', usecols = range(7))
        instances = {"activity": activity, "file" : file}

        for i , col in enumerate(time_series, start=1):
            instances[f'min{i}'] = df[col].min()
            instances[f'max{i}'] = df[col].max()
            instances[f'mean{i}'] = df[col].mean()
            instances[f'median{i}'] = df[col].median()
            instances[f'std{i}'] = df[col].std()
            instances[f"first_quartile{i}"] = df[col].quantile(0.25)
            instances[f"third_quartile{i}"] = df[col].quantile(0.75)
        all.append(instances)
feature_df = pd.DataFrame(all)
feature_df.head()


,activity,file,min1,max1,mean1,median1,std1,first_quartile1,third_quartile1,min2,...,std5,first_quartile5,third_quartile5,min6,max6,mean6,median6,std6,first_quartile6,third_quartile6
0,bending1,dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1,dataset2.csv,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1,dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1,dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1,dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [29]:
feature_col = []

for i in range (1, 7):
    feature_col.extend([
        f'min{i}',
        f'max{i}',
        f'mean{i}',
        f'median{i}',
        f'std{i}',
        f'first_quartile{i}',
        f'third_quartile{i}',
    ])

X_feature =feature_df[feature_col]


std = X_feature.std(axis=0, ddof=1)

n = X_feature.shape[0]

boot_std = pd.DataFrame(index = range(1000), columns=X_feature.columns)

for b in range(1000):
    boot_sample = X_feature.sample(n=n, replace=True)
    boot_std.loc[b] = boot_sample.std(axis=0, ddof=1)

std_result = pd.DataFrame({
    "Feature": feature_col,
    "Estimated STD": std.values,
    "90% CI Lower": boot_std.quantile(0.05).values,
    "90% CI Upper": boot_std.quantile(0.95).values,

})
std_result

,Feature,Estimated STD,90% CI Lower,90% CI Upper
0,min1,9.569975,8.239422,10.722277
1,max1,4.394362,3.337644,5.278152
2,mean1,5.335718,4.719511,5.889199
3,median1,5.440054,4.808723,5.988758
4,std1,1.772153,1.565173,1.949092
5,first_quartile1,6.153590,5.544905,6.63529
6,third_quartile1,5.138925,4.345439,5.836537
7,min2,0.000000,0.0,0.0
8,max2,5.062729,4.62881,5.395674
9,mean2,1.574164,1.396628,1.702171


#### iv. Select Features

In general, I choose mean, median and standard deviation. The mean tells us the average sensor value over the whole time window and also captures the general level of the movement. The median serve same aspect and function as the mean but is more robust to outliers which would affect the mean. Standard deviation measures variability. In our case it is important because activities like walking and cycling usually produce more variations, on the other hand sitting and lying may have lower variations.

## 2. ISLR 3.7.4

### (a) Linear Train

I think the cubic regression will have a slightly smaller RSS then linear regression. Because we can actually say linear model is a special case of cubic regression and with more parameters, cubic model is more flexible and thus fit at least as good as the linear model. Since in our case the actual relationship is linear, the RSS for cubic will be lower because it fit more random noise.

### (b) Linear Test

In the test case, the linear regression will have lower RSS. Because our actual relationship is linear, our cubic regression will fit noise and cause overfitting.

### (c) Not Linear Train

Here the cubic regression will have lower or equal RSS to linear model because in general the cubic regression is more flexible than linear regression. Cubic regression will fit at least as good as the linear model, regardless of the actual relationship.

### (d) Not Linear Testing

In this case we don't have enough information to tell. Since we de do not know how far away from linear is the actual relationship, if it is closer to cubic, then cubic model may have less RSS; on the other hand if it is just a little off from linear, then we may have the same situation in (b) where cubic may overfit.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 